In [0]:
# Instala as bibliotecas necessárias
%pip install azure-storage-file-datalake azure-identity pandas pyodbc sqlalchemy python-dotenv openpyxl pymssql

In [0]:
# Imports e Configuração

import os
import urllib
import pandas as pd
import pymssql
from io import BytesIO
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from sqlalchemy import create_engine, text

load_dotenv(dotenv_path="/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/.env")

print("✅ Imports carregados com sucesso!")

In [0]:
# Conexão com o Data Lake (ADLS Gen2)

def get_adls_client():
    credential = ClientSecretCredential(
        tenant_id=os.getenv("tenant_id"),
        client_id=os.getenv("client_id"),
        client_secret=os.getenv("client_secret")
    )

    account_name = os.getenv("storage_account_name")
    account_url  = f"https://{account_name}.dfs.core.windows.net"

    return DataLakeServiceClient(account_url=account_url, credential=credential)

print("✅ Função de conexão com ADLS criada!")

In [0]:
# Mount do ADLS Gen2 no Databricks
def montar_data_lake():
    storage_account = os.getenv("storage_account_name")
    container       = os.getenv("container_name")
    client_id_val   = os.getenv("client_id")
    tenant_id_val   = os.getenv("tenant_id")
    client_secret_val = os.getenv("client_secret")

    configs = {
        "fs.azure.account.auth.type": "OAuth",
        "fs.azure.account.oauth.provider.type": 
            "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        "fs.azure.account.oauth2.client.id": client_id_val,
        "fs.azure.account.oauth2.client.secret": client_secret_val,
        "fs.azure.account.oauth2.client.endpoint": 
            f"https://login.microsoftonline.com/{tenant_id_val}/oauth2/token"
    }

    mount_point = f"/mnt/{container}"

    # Verifica se já está montado
    already_mounted = any(
        mount.mountPoint == mount_point 
        for mount in dbutils.fs.mounts()
    )

    if not already_mounted:
        dbutils.fs.mount(
            source=f"abfss://{container}@{storage_account}.dfs.core.windows.net/",
            mount_point=mount_point,
            extra_configs=configs
        )
        print(f"✅ Data Lake montado em: {mount_point}")
    else:
        print(f"✅ Data Lake já estava montado em: {mount_point}")

    # Lista arquivos após montar
    print("\n📁 Arquivos disponíveis:")
    arquivos = dbutils.fs.ls(mount_point)
    for arquivo in arquivos:
        print(f"   - {arquivo.name}")

montar_data_lake()

In [0]:
# Listar Arquivos do Container

def listar_arquivos():
    client_id_val     = os.getenv("client_id")
    tenant_id_val     = os.getenv("tenant_id")
    client_secret_val = os.getenv("client_secret")
    container         = os.getenv("container_name")

    if not all([client_id_val, tenant_id_val, client_secret_val, container]):
        raise ValueError("❌ Uma ou mais variáveis do .env estão vazias. Verifique o arquivo .env")

    client    = get_adls_client()
    fs_client = client.get_file_system_client(file_system=container)
    arquivos  = [p.name for p in fs_client.get_paths() if not p.is_directory]
    return arquivos

arquivos = listar_arquivos()
print(f"📁 Arquivos encontrados: {len(arquivos)}\n")
for arquivo in arquivos:
    print(" -", arquivo)

In [0]:
# Conexão com SQL Server

def get_engine():
    host     = os.getenv("jdbc_hostname")
    database = os.getenv("jdbc_database")
    username = os.getenv("jdbc_username")
    password = os.getenv("jdbc_password")

    if not all([host, database, username, password]):
        raise ValueError("❌ Uma ou mais variáveis SQL do .env estão vazias.")

    # Usa pymssql — não precisa de ODBC Driver
    engine = create_engine(
        f"mssql+pymssql://{username}:{password}@{host}/{database}"
    )
    return engine

print("✅ Função de conexão com SQL Server criada via pymssql!")

In [0]:
# Teste JDBC (CREATE/INSERT)

def testar_conexao_sql():
    engine = get_engine()

    with engine.connect() as conn:
        # Cria schema squad3 se não existir
        conn.execute(text("""
            IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = 'squad3')
            BEGIN
                EXEC('CREATE SCHEMA squad3')
            END
        """))

        # Testa CREATE
        conn.execute(text("""
            IF OBJECT_ID('squad3.teste_conexao', 'U') IS NOT NULL
                DROP TABLE squad3.teste_conexao;

            CREATE TABLE squad3.teste_conexao (
                id INT,
                mensagem VARCHAR(100),
                dt_teste DATETIME
            );
        """))

        # Testa INSERT
        conn.execute(text("""
            INSERT INTO squad3.teste_conexao VALUES (
                1,
                'Conexão com sucesso',
                GETDATE()
            );
        """))
        conn.commit()

        # Verifica resultado
        resultado = conn.execute(
            text("SELECT * FROM squad3.teste_conexao")
        ).fetchall()

        print("✅ Conexão JDBC testada com sucesso!")
        print(f"   Permissão de CREATE : ✅")
        print(f"   Permissão de INSERT : ✅")
        print(f"   Registro inserido   : {resultado}")

testar_conexao_sql()

In [0]:
# Função Ler parquet do Data Lake

def ler_csv(caminho_arquivo, separador=","):
    client_id_val     = os.getenv("client_id")
    tenant_id_val     = os.getenv("tenant_id")
    client_secret_val = os.getenv("client_secret")
    account_name      = os.getenv("storage_account_name")
    container         = os.getenv("container_name")

    if not all([client_id_val, tenant_id_val, client_secret_val, account_name, container]):
        raise ValueError("❌ Uma ou mais variáveis do .env estão vazias. Verifique o arquivo .env")

    credential  = ClientSecretCredential(
        tenant_id=tenant_id_val,
        client_id=client_id_val,
        client_secret=client_secret_val
    )
    account_url = f"https://{account_name}.dfs.core.windows.net"
    client      = DataLakeServiceClient(account_url=account_url, credential=credential)
    fs_client   = client.get_file_system_client(file_system=container)
    file_client = fs_client.get_file_client(caminho_arquivo)

    conteudo = file_client.download_file().readall()
    return pd.read_csv(BytesIO(conteudo), sep=separador)


def ler_parquet(caminho_arquivo):                        # ← ADICIONADO
    client_id_val     = os.getenv("client_id")
    tenant_id_val     = os.getenv("tenant_id")
    client_secret_val = os.getenv("client_secret")
    account_name      = os.getenv("storage_account_name")
    container         = os.getenv("container_name")

    if not all([client_id_val, tenant_id_val, client_secret_val, account_name, container]):
        raise ValueError("❌ Uma ou mais variáveis do .env estão vazias. Verifique o arquivo .env")

    credential  = ClientSecretCredential(
        tenant_id=tenant_id_val,
        client_id=client_id_val,
        client_secret=client_secret_val
    )
    account_url = f"https://{account_name}.dfs.core.windows.net"
    client      = DataLakeServiceClient(account_url=account_url, credential=credential)
    fs_client   = client.get_file_system_client(file_system=container)
    file_client = fs_client.get_file_client(caminho_arquivo)

    conteudo = file_client.download_file().readall()
    return pd.read_parquet(BytesIO(conteudo))            # ← diferença aqui


print("✅ Funções ler_csv e ler_parquet criadas!")

In [0]:
# Ler Arquivo do Data Lake

df = ler_parquet("vendas_raw/2026/02/21/112200/ecommerce_pedidos.parquet")   

print(f"✅ Arquivo lido com sucesso! Shape: {df.shape}")

In [0]:
# Análise Exploratória

print("=" * 50)
print("📊 ANÁLISE EXPLORATÓRIA")
print("=" * 50)

print(f"\n📐 Shape: {df.shape}")
print(f"   {df.shape[0]} linhas | {df.shape[1]} colunas")

print("\n📋 Colunas e tipos de dados:")
print(df.dtypes)

print("\n❓ Valores nulos por coluna:")
print(df.isnull().sum())

print(f"\n🔁 Duplicatas: {df.duplicated().sum()}")

print("\n📈 Estatísticas descritivas:")
df.describe()

In [0]:
# Primeiras linhas

print("👀 Primeiras 10 linhas:")
df.head(10)

In [0]:
# Tratamento básico de dados 


print("=" * 50)
print("🔧 TRATAMENTOS — ecommerce_pedidos")
print("=" * 50)

# ══════════════════════════════════════
# 1. DATAS — converter para datetime
# ══════════════════════════════════════
colunas_data = ['dt_pedido', 'dt_ultima_atualizacao_status']

for col in colunas_data:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"✅ {col} convertida para datetime")

# ══════════════════════════════════════
# 2. DUPLICATAS — verificar id_pedido duplicado
# ══════════════════════════════════════
total_antes = len(df)
duplicatas = df.duplicated(subset=['id_pedido']).sum()
df = df.drop_duplicates(subset=['id_pedido'])
print(f"\n✅ Duplicatas em id_pedido removidas: {duplicatas} linhas")
print(f"   {total_antes} → {len(df)} linhas")

# ══════════════════════════════════════
# 3. INCONSISTÊNCIA — id_cliente = id_pedido
# ══════════════════════════════════════
inconsistentes = (df['id_cliente'] == df['id_pedido']).sum()
print(f"\n⚠️  Registros onde id_cliente == id_pedido: {inconsistentes}")
# Apenas registra — não remove, pois pode ser dado válido
# Documente no PR se encontrar muitos casos

# ══════════════════════════════════════
# 4. NULOS — verificar e tratar
# ══════════════════════════════════════
df['status_pedido']      = df['status_pedido'].fillna('desconhecido')
df['metodo_pagamento']   = df['metodo_pagamento'].fillna('desconhecido')
df['valor_total']        = df['valor_total'].fillna(0)
df['valor_frete']        = df['valor_frete'].fillna(0)
print("\n✅ Nulos tratados")

# ══════════════════════════════════════
# 5. PADRONIZAR TEXTO
# ══════════════════════════════════════

# status_pedido → maiúsculo e sem espaços extras
df['status_pedido'] = df['status_pedido'].str.strip().str.upper()
print("\n✅ status_pedido padronizado:")
print(df['status_pedido'].value_counts())

# metodo_pagamento → minúsculo e sem espaços extras
df['metodo_pagamento'] = df['metodo_pagamento'].str.strip().str.lower()
print("\n✅ metodo_pagamento padronizado:")
print(df['metodo_pagamento'].value_counts())

# ══════════════════════════════════════
# 6. VALORES — garantir que não são negativos
# ══════════════════════════════════════
for col in ['valor_total', 'valor_frete']:
    negativos = (df[col] < 0).sum()
    if negativos > 0:
        df[col] = df[col].abs()
        print(f"\n⚠️  {col} — {negativos} valores negativos corrigidos")
    else:
        print(f"\n✅ {col} — nenhum valor negativo encontrado")

# ══════════════════════════════════════
# 7. TIPOS FINAIS — garantir tipos corretos
# ══════════════════════════════════════
df['id_pedido']            = df['id_pedido'].astype(str)
df['id_cliente']           = df['id_cliente'].astype(str)
df['id_endereco_entrega']  = df['id_endereco_entrega'].astype(str)
df['valor_total']          = df['valor_total'].astype(float)
df['valor_frete']          = df['valor_frete'].astype(float)
print("\n✅ Tipos garantidos")

# ══════════════════════════════════════
# RESULTADO FINAL
# ══════════════════════════════════════
print(f"\n📐 Shape final: {df.shape}")
print(f"   {df.shape[0]} linhas | {df.shape[1]} colunas")
print("\n👀 Amostra final:")
df.head()



In [0]:
# Salvar no SQL Server

def salvar_tabela(df, nome_tabela, modo="replace"):
    engine = get_engine()

    # Cria o schema squad3 se não existir
    with engine.connect() as conn:
        conn.execute(text("""
            IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = 'squad3')
            BEGIN
                EXEC('CREATE SCHEMA squad3')
            END
        """))
        conn.commit()

    df.to_sql(
        name=nome_tabela,
        con=engine,
        schema="squad3",
        if_exists="replace",
        index=False
    )
    print(f"✅ Tabela squad3.{nome_tabela} salva com sucesso! ({len(df)} linhas)")

# Salva a tabela
salvar_tabela(df, nome_tabela="ecommerce_pedidos")

In [0]:
# Verificar Dados Salvos

def consultar_tabela(nome_tabela):
    engine = get_engine()
    df_resultado = pd.read_sql(
        f"SELECT * FROM squad3.{nome_tabela}",
        con=engine
    )
    return df_resultado

df_verificacao = consultar_tabela("ecommerce_pedidos")

print(f"✅ Verificação concluída!")
print(f"   Linhas no banco: {len(df_verificacao)}")
df_verificacao.head()